# Lezione 3 — Classificazione, regressione e Functional API di Keras

Corso "Fondamenti di TensorFlow da zero" — DIH DATAlife (Hub di Innovazione Digitale Europeo, EDIH), a partire dalla certificazione EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals) "Fondamenti di TensorFlow".

## Obiettivi di questa lezione

- Distinguere tra problemi di classificazione e di regressione e scegliere la funzione di perdita corretta per ciascuno
- Caricare e preparare dati tabulari in formato CSV (Comma-Separated Values, valori separati da virgole) per addestrare un modello
- Costruire modelli con la Functional API di Keras per architetture non sequenziali
- Riconoscere le differenze chiave tra TensorFlow 1.x e TensorFlow 2.0

Questo notebook accompagna il capitolo 05 del manuale del corso. Esegui le celle in ordine: ogni blocco di codice è commentato in modo che tu comprenda cosa fa e perché.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

print("Versión de TensorFlow:", tf.__version__)
print("Ejecución eager activada:", tf.executing_eagerly())

## 1. Classificazione contro regressione

Due domande diverse richiedono due modi diversi di addestrare un modello:

- **Classificazione**: l'uscita è una categoria di un insieme chiuso ("che indumento è?", "il tumore è benigno o maligno?")
- **Regressione**: l'uscita è un numero continuo ("quanti chilometri per litro fa questa auto?")

Questa differenza cambia l'ultimo strato della rete e, soprattutto, la funzione di perdita:

| Problema | Ultimo strato | Perdita |
|---|---|---|
| Classificazione binaria | 1 neurone, attivazione `sigmoid` | `binary_crossentropy` |
| Classificazione multiclasse | *n* neuroni (logit o `softmax`) | `sparse_categorical_crossentropy` |
| Regressione | 1 neurone, senza attivazione (lineare) | `mean_squared_error` o `mean_absolute_error` |

### 1.1 Esempio di classificazione: Fashion MNIST

Addestriamo un classificatore multiclasse con il dataset Fashion MNIST (10 categorie di abbigliamento, immagini di 28x28 pixel in scala di grigi).

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = ['Camiseta', 'Pantalon', 'Jersey', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botin']

# Normalizamos los píxeles al rango 0-1 (estaban en 0-255)
train_images = train_images / 255.0
test_images = test_images / 255.0

print("Imágenes de entrenamiento:", train_images.shape)
print("Etiquetas de entrenamiento:", train_labels.shape)

In [ ]:
# Modelo Sequential: una pila simple de capas, suficiente para este problema
modelo_clasificacion = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),   # aplana la imagen 28x28 a un vector de 784
    tf.keras.layers.Dense(128, activation='relu'),   # capa oculta
    tf.keras.layers.Dense(10)                        # 10 salidas (logits), una por clase
])

# Clasificación multiclase -> sparse_categorical_crossentropy
# from_logits=True porque la última capa no tiene activación softmax
modelo_clasificacion.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

# Entrenamos pocas épocas para que la celda se ejecute rápido en clase
modelo_clasificacion.fit(train_images, train_labels, epochs=5, verbose=1)

In [ ]:
test_loss, test_acc = modelo_clasificacion.evaluate(test_images, test_labels, verbose=0)
print(f"Precisión sobre el conjunto de prueba: {test_acc:.2%}")

# Convertimos los logits en probabilidades con una capa Softmax para interpretar la salida
modelo_probabilidades = tf.keras.Sequential([modelo_clasificacion, tf.keras.layers.Softmax()])
prediccion = modelo_probabilidades.predict(test_images[:1], verbose=0)
print("Clase predicha:", class_names[np.argmax(prediccion[0])])
print("Clase real:", class_names[test_labels[0]])

## 1. Classificazione rispetto a regressione

Due domande diverse richiedono due modi diversi di addestrare un modello:

- **Classificazione**: l'output è una categoria da un insieme chiuso ("quale indumento è?", "il tumore è benigno o maligno?")
- **Regressione**: l'output è un numero continuo ("quanti chilometri per litro fa questa macchina?")

Questa differenza cambia l'ultimo strato della rete e, soprattutto, la funzione di perdita:

| Problema | Ultimo strato | Perdita |
|---|---|---|
| Classificazione binaria | 1 neurone, attivazione `sigmoid` | `binary_crossentropy` |
| Classificazione multiclasse | *n* neuroni (logits o `softmax`) | `sparse_categorical_crossentropy` |
| Regressione | 1 neurone, nessuna attivazione (lineare) | `mean_squared_error` o `mean_absolute_error` |

### 1.1 Esempio di classificazione: Fashion MNIST

Andiamo ad addestrare un classificatore multiclasse con il dataset Fashion MNIST (10 categorie di abbigliamento, immagini di 28x28 pixel in scala di grigi).

In [ ]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
nombres_columnas = ['MPG', 'Cilindros', 'Desplazamiento', 'Caballos', 'Peso',
                     'Aceleracion', 'Año', 'Origen']

dataset_crudo = pd.read_csv(url, names=nombres_columnas, na_values='?',
                             comment='\t', sep=' ', skipinitialspace=True)

dataset = dataset_crudo.dropna()  # eliminamos filas con valores desconocidos

# 'Origen' es categórica (1=EEUU, 2=Europa, 3=Japón): la convertimos a one-hot
dataset['Origen'] = dataset['Origen'].map({1: 'EEUU', 2: 'Europa', 3: 'Japon'})
dataset = pd.get_dummies(dataset, columns=['Origen'], prefix='', prefix_sep='')

dataset.tail()

In [ ]:
# División en entrenamiento (80%) y prueba (20%)
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

# Separamos la etiqueta (lo que queremos predecir) de las características
train_features = train_dataset.copy()
test_features = test_dataset.copy()
train_labels = train_features.pop('MPG')
test_labels = test_features.pop('MPG')

# Capa de normalización: se "adapta" a las estadísticas del set de entrenamiento
normalizador = tf.keras.layers.Normalization(axis=-1)
normalizador.adapt(np.array(train_features))

In [ ]:
# Modelo de regresión: última capa con una neurona y sin activación (salida lineal)
modelo_regresion = tf.keras.Sequential([
    normalizador,
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])

# Regresión -> mean_absolute_error (o mean_squared_error), nunca accuracy
modelo_regresion.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='mean_absolute_error')

historial = modelo_regresion.fit(
    train_features, train_labels,
    validation_split=0.2,
    verbose=0, epochs=100)

error_test = modelo_regresion.evaluate(test_features, test_labels, verbose=0)
print(f"Error absoluto medio en el conjunto de prueba: {error_test:.2f} MPG")

## 2. Caricare e preparare dati CSV

La maggior parte dei dataset tabulari reali mescola colonne numeriche e categoriche, e spesso ha scale molto diverse. Vediamo due tecniche:

1. Normalizzare le colonne numeriche con `tf.keras.layers.Normalization`
2. Convertire le colonne categoriche in one-hot con `StringLookup` + `CategoryEncoding`, usando il dataset del Titanic come esempio.

In [ ]:
titanic = pd.read_csv("https://storage.googleapis.com/tf-datasets/titanic/train.csv")
titanic_features = titanic.copy()
titanic_labels = titanic_features.pop('survived')  # 0 = no sobrevivió, 1 = sobrevivió

titanic.head()

In [ ]:
# Creamos una entrada simbólica (Input) por cada columna del CSV, respetando su tipo
inputs = {}
for nombre, columna in titanic_features.items():
    dtype = tf.string if columna.dtype == object else tf.float32
    inputs[nombre] = tf.keras.Input(shape=(1,), name=nombre, dtype=dtype)

# Las columnas numéricas se concatenan y se pasan por una capa de normalización
entradas_numericas = {nombre: entrada for nombre, entrada in inputs.items()
                       if entrada.dtype == tf.float32}
x = tf.keras.layers.Concatenate()(list(entradas_numericas.values()))
normalizador_titanic = tf.keras.layers.Normalization()
normalizador_titanic.adapt(np.array(titanic[entradas_numericas.keys()]))
numericas_procesadas = normalizador_titanic(x)

preprocesado = [numericas_procesadas]

# Las columnas de texto se convierten en índices (StringLookup) y luego en one-hot (CategoryEncoding)
for nombre, entrada in inputs.items():
    if entrada.dtype == tf.float32:
        continue
    lookup = tf.keras.layers.StringLookup(vocabulary=np.unique(titanic_features[nombre]))
    one_hot = tf.keras.layers.CategoryEncoding(num_tokens=lookup.vocabulary_size())
    x = one_hot(lookup(entrada))
    preprocesado.append(x)

# Este modelo solo hace preprocesamiento: no predice nada todavía
preprocesado_concatenado = tf.keras.layers.Concatenate()(preprocesado)
titanic_preprocesamiento = tf.keras.Model(inputs, preprocesado_concatenado)
print("Preprocesamiento definido con", len(inputs), "columnas de entrada")

Nota una cosa importante: per definire questo preprocessamento abbiamo usato `tf.keras.Input` e abbiamo concatenato strati chiamandoli come funzioni su tensori (`normalizador_titanic(x)`, `lookup(entrada)`). Questa è esattamente la **Functional API**, che vediamo in dettaglio nella sezione seguente: ne abbiamo bisogno qui perché il modello ha tanti ingressi quante colonne ha il CSV, qualcosa che la Sequential API non può esprimere.

In [ ]:
# Construimos el modelo completo: preprocesamiento + red densa de clasificación binaria
def construir_modelo_titanic(cabecera_preprocesamiento, entradas):
    cuerpo = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    entradas_procesadas = cabecera_preprocesamiento(entradas)
    resultado = cuerpo(entradas_procesadas)
    modelo = tf.keras.Model(entradas, resultado)
    modelo.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                    optimizer=tf.keras.optimizers.Adam())
    return modelo

modelo_titanic = construir_modelo_titanic(titanic_preprocesamiento, inputs)

# Los modelos con entradas nombradas esperan un diccionario de arrays, no una tabla de pandas
titanic_features_dict = {nombre: np.array(valor) for nombre, valor in titanic_features.items()}
modelo_titanic.fit(x=titanic_features_dict, y=titanic_labels, epochs=10, verbose=0)
print("Entrenamiento completado")

## 3. La Functional API di Keras

La Sequential API impila strati uno dietro l'altro: un ingresso, un percorso lineare, un'uscita. Non appena hai bisogno di **più ingressi**, **più uscite** o connessioni che non seguono quell'ordine, passi alla Functional API.

L'idea chiave: ogni strato è una funzione che si applica su un tensore.

```python
entrada = keras.Input(shape=(784,))
x = layers.Dense(64, activation="relu")(entrada)
x = layers.Dense(64, activation="relu")(x)
salida = layers.Dense(10)(x)
modelo = keras.Model(inputs=entrada, outputs=salida)
```

Costruiamo un modello con **due ingressi indipendenti** che si combinano prima dell'uscita finale: un caso che la Sequential API non può risolvere.

In [ ]:
# Ejemplo: un modelo que combina datos numéricos y datos categóricos ya codificados
entrada_numerica = tf.keras.Input(shape=(8,), name="datos_numericos")
entrada_categorica = tf.keras.Input(shape=(4,), name="datos_categoricos")

# Cada entrada sigue su propio camino antes de combinarse
rama_numerica = tf.keras.layers.Dense(32, activation='relu')(entrada_numerica)
rama_numerica = tf.keras.layers.Dense(16, activation='relu')(rama_numerica)

rama_categorica = tf.keras.layers.Dense(16, activation='relu')(entrada_categorica)

# concatenate une las dos ramas en un único vector de características
combinado = tf.keras.layers.concatenate([rama_numerica, rama_categorica])
salida = tf.keras.layers.Dense(1, activation='sigmoid')(combinado)

modelo_multientrada = tf.keras.Model(
    inputs=[entrada_numerica, entrada_categorica],
    outputs=salida,
    name="modelo_dos_entradas")

modelo_multientrada.summary()

In [ ]:
# Datos de ejemplo (aleatorios) solo para comprobar que el modelo funciona de punta a punta
datos_numericos_ejemplo = np.random.random((100, 8)).astype("float32")
datos_categoricos_ejemplo = np.random.randint(2, size=(100, 4)).astype("float32")
etiquetas_ejemplo = np.random.randint(2, size=(100, 1)).astype("float32")

modelo_multientrada.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo_multientrada.fit(
    {"datos_numericos": datos_numericos_ejemplo, "datos_categoricos": datos_categoricos_ejemplo},
    etiquetas_ejemplo,
    epochs=3, verbose=1)

### 1.2 Esempio di regressione: consumo di combustibile (Auto MPG)

Ora risolviamo un problema di regressione: prevedere i chilometri per litro (MPG, *miles per gallon*, una misura del consumo) di automobili dagli anni settanta in base alle loro caratteristiche (cilindri, peso, cavalli vapore, tra gli altri).

## 4. TensorFlow 1.x contro TensorFlow 2.0

Il codice che abbiamo scritto in tutto questo corso usa l'**esecuzione eager**: ogni operazione viene calcolata all'istante, come nel Python normale. Così funzionava TensorFlow **prima** della versione 2.0, con sessioni e grafi:

```python
# Stile TensorFlow 1.x (non eseguirlo, solo per confronto)
import tensorflow as tf
a = tf.constant(2)
b = tf.constant(3)
c = tf.add(a, b)          # non calcola ancora nulla, definisce solo il grafo

with tf.Session() as sess:
    resultado = sess.run(c)  # qui sì viene eseguito il grafo
    print(resultado)
```

Confrontalo con lo stesso calcolo in TensorFlow 2.x:

In [ ]:
a = tf.constant(2)
b = tf.constant(3)
c = tf.add(a, b)
print(c)  # el resultado ya está disponible, sin sesión ni sess.run()

Cambiamenti chiave portati da TensorFlow 2.0:

- **Esecuzione eager di default**: senza `tf.Session` né `tf.placeholder`, ogni tensore ha il suo valore immediatamente
- **Keras come API ufficiale di alto livello**: prima convivevano varie forme di costruire modelli, ora `tf.keras` è quella raccomandata
- **`tf.function`**: converte una funzione Python in un grafo ottimizzato quando hai bisogno di più prestazioni, senza tornare alle sessioni
- **API più semplice e coerente**, con meno codice ripetitivo
- **Un unico formato di salvataggio** (`SavedModel`) compatibile con TensorFlow Serving, TensorFlow Lite e TensorFlow.js

`tf.function` è utile, per esempio, in un ciclo di addestramento personalizzato con `tf.GradientTape` (lo strumento che calcola i gradienti automaticamente):

In [ ]:
# Ejemplo de un paso de entrenamiento manual, típico de TensorFlow 2.x
optimizador = tf.keras.optimizers.Adam()
funcion_perdida = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@tf.function  # convierte esta función en un grafo optimizado la primera vez que se llama
def paso_de_entrenamiento(modelo, imagenes, etiquetas):
    with tf.GradientTape() as tape:
        predicciones = modelo(imagenes, training=True)
        perdida = funcion_perdida(etiquetas, predicciones)
    gradientes = tape.gradient(perdida, modelo.trainable_variables)
    optimizador.apply_gradients(zip(gradientes, modelo.trainable_variables))
    return perdida

# Probamos el paso con un lote pequeño de Fashion MNIST
lote_imagenes = train_images[:32].astype("float32")
lote_etiquetas = train_labels[:32]
perdida_obtenida = paso_de_entrenamiento(modelo_clasificacion, lote_imagenes, lote_etiquetas)
print("Pérdida del lote de prueba:", float(perdida_obtenida))

## Riepilogo

- Classificazione: uscita categorica, perdite tipo `*_crossentropy`, metrica `accuracy`
- Regressione: uscita numerica continua, perdite `mean_squared_error` o `mean_absolute_error`
- I dati CSV richiedono preprocessamento: separare le etichette, normalizzare le colonne numeriche, convertire le categoriche in one-hot
- La Sequential API serve per pile semplici di strati; la Functional API è necessaria per più ingressi, più uscite o connessioni non lineari
- TensorFlow 2.0 ha sostituito sessioni e grafi con l'esecuzione eager, con Keras come API ufficiale e `tf.function` disponibile quando serve prestazione da grafo

Continua con il notebook di pratica di questa lezione: `clase03_practica_clasificacion_regresion_keras.ipynb`.